# LinkedIn MCP

In [1]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from google.oauth2 import id_token as oauth2_id_token
import google.auth.transport.requests

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
load_dotenv()

True

In [4]:
# Connect to LinkedIn MCP
# Run the MCP server from its own directory so uv finds the project (avoids "Connection closed").
# Prerequisites (from linkedin-mcp-server dir):
#   uv run linkedin-mcp-server --get-session   # create LinkedIn session once
#   uv run playwright install chromium         # if you see "Failed to start browser"

uv_path = shutil.which("uv") or "/opt/homebrew/bin/uv"
linkedin_server_dir = os.path.expanduser("~/dev/linkedin-mcp-server")
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"
env["TRANSPORT"] = "stdio"

client = MultiServerMCPClient(
    {
        "linkedin": {
            "command": uv_path,
            "transport": "stdio",
            "args": ["--directory", linkedin_server_dir, "run", "linkedin-mcp-server"],
            "cwd": linkedin_server_dir,
            "env": env,
        }
    }
)
tools = await client.get_tools()

In [2]:
LINKEDIN_MCP_SERVICE_URL = "https://linkedin-mcp-server-660196542212.europe-west1.run.app"

# Set service account credentials (same as CH MCP)
credentials_path = project_root / ".secrets" / "fionaa-service-acct.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())

# Fetch IAM identity token for Cloud Run (requires roles/run.invoker on the service account)
request = google.auth.transport.requests.Request()
token = oauth2_id_token.fetch_id_token(request, LINKEDIN_MCP_SERVICE_URL)

client = MultiServerMCPClient(
    {
        "linkedin": {
            "transport": "streamable_http",
            "url": f"{LINKEDIN_MCP_SERVICE_URL}/mcp",
            "headers": {"Authorization": f"Bearer {token}"},
        }
    }
)
tools = await client.get_tools()
tools

[StructuredTool(name='get_person_profile', description='Get a specific person\'s LinkedIn profile.\n\nArgs:\n    linkedin_username: LinkedIn username (e.g., "stickerdaniel", "williamhgates")\n    ctx: FastMCP context for progress reporting\n\nReturns:\n    Structured data from the person\'s profile including:\n    - linkedin_url, name, location, about, open_to_work\n    - experiences: List of work history (position_title, institution_name,\n      linkedin_url, from_date, to_date, duration, location, description)\n    - educations: List of education (institution_name, degree, linkedin_url,\n      from_date, to_date, description)\n    - interests: List of interests with category (company, group, school,\n      newsletter, influencer) and linkedin_url\n    - accomplishments: List of accomplishments (category, title)\n    - contacts: List of contact info (type: email/phone/website/linkedin/\n      twitter/birthday/address, value, label)', args_schema={'properties': {'linkedin_username': {'

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="you are a helpful assistant that can answer questions about the companies on linked in",
)
# # MCP tools are async-only; use ainvoke so the graph runs tools asynchronously
# result = await agent.ainvoke(
#     {"messages": [HumanMessage(content="Tell me about the directors of company Goodmans Consulting")]}
# )
# result

In [ ]:
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about Steve Goodman who lives in London UK, works at Intact Financial Corperation")]}
)
result